# Imports

In [1]:
import torch
import sys


from torch_geometric.datasets import Planetoid

c:\faculdade\Tabalho-Final-XAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import random
import torch.nn.functional as F
from torch_geometric.nn import  HANConv
from torch_geometric.datasets import DBLP
import os
import pandas as pd

In [3]:
from itertools import combinations
from tqdm import tqdm
import re
from scipy.stats import spearmanr

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [5]:
import numpy as np
import warnings
import pickle
from pathlib import Path

In [6]:
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import PGExplainer
from torch_geometric.explain.config import ModelConfig

# Dados

In [7]:
def load_dataset(name):

    if name in [
        "Cora",
        "CiteSeer",
        "PubMed"
    ]:

        dataset = Planetoid(
            root=f"data/{name}",
            name=name
        )

        return dataset

    elif name == "DBLP":

        dataset = DBLP(
            root="data/DBLP"
        )

        return dataset

    else:

        raise ValueError(
            f"Dataset {name} não suportado."
        )

DBLP

In [8]:
from torch_geometric.data import HeteroData


In [9]:
DBdataset = DBLP(
    root='data/DBLP'
)

In [10]:
dblp = DBdataset[0]


dblp_new = HeteroData()

# nós
dblp_new['author'].x = dblp['author'].x
dblp_new['author'].y = dblp['author'].y
dblp_new['author'].train_mask = dblp['author'].train_mask
dblp_new['author'].val_mask = dblp['author'].val_mask
dblp_new['author'].test_mask = dblp['author'].test_mask

dblp_new['paper'].x = dblp['paper'].x

dblp_new['term'].x = dblp['term'].x

In [11]:
dblp_new['author', 'to', 'paper'].edge_index = \
    dblp['author', 'to', 'paper'].edge_index

dblp_new['paper', 'to', 'author'].edge_index = \
    dblp['paper', 'to', 'author'].edge_index

dblp_new['paper', 'to', 'term'].edge_index = \
    dblp['paper', 'to', 'term'].edge_index

dblp_new['term', 'to', 'paper'].edge_index = \
    dblp['term', 'to', 'paper'].edge_index

In [12]:
print(dblp_new.metadata())

(['author', 'paper', 'term'], [('author', 'to', 'paper'), ('paper', 'to', 'author'), ('paper', 'to', 'term'), ('term', 'to', 'paper')])


In [13]:
print(dblp_new.node_types)

['author', 'paper', 'term']


In [14]:
print(dblp_new.edge_types)

[('author', 'to', 'paper'), ('paper', 'to', 'author'), ('paper', 'to', 'term'), ('term', 'to', 'paper')]


In [15]:
print(dblp_new["author"])

{'x': tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]), 'y': tensor([2, 2, 3,  ..., 0, 0, 0]), 'train_mask': tensor([False, False, False,  ..., False, False, False]), 'val_mask': tensor([False, False,  True,  ..., False, False, False]), 'test_mask': tensor([ True,  True, False,  ...,  True,  True,  True])}


In [16]:
for node_type in dblp_new.node_types:

    print()
    print(node_type)
    print(dblp_new[node_type])


author
{'x': tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]), 'y': tensor([2, 2, 3,  ..., 0, 0, 0]), 'train_mask': tensor([False, False, False,  ..., False, False, False]), 'val_mask': tensor([False, False,  True,  ..., False, False, False]), 'test_mask': tensor([ True,  True, False,  ...,  True,  True,  True])}

paper
{'x': tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])}

term
{'x': tensor([[-0.6924, -0.4659,  1.1540,  ...,  0.9178,  0.1995, -0.6360],
        [ 1.2031, -0.4003,  0.0740,  ...,  1.3262, -0.3325,  0.8198],
        [ 0.3748,  0.5731,  0.4802,  ...,  1.1522,  0.6010,

In [17]:
for node_type in dblp_new.node_types:

    print(node_type)
    print("num_nodes =", dblp_new[node_type].num_nodes)

    if 'x' in dblp_new[node_type]:
        print("x.shape =", dblp_new[node_type].x.shape)

    print()

author
num_nodes = 4057
x.shape = torch.Size([4057, 334])

paper
num_nodes = 14328
x.shape = torch.Size([14328, 4231])

term
num_nodes = 7723
x.shape = torch.Size([7723, 50])



# Modelo

In [18]:
class HAN(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        metadata,
        heads=8,
        dropout=0.5
    ):
        super().__init__()

        self.conv = HANConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            metadata=metadata,
            heads=heads
        )

        self.dropout = torch.nn.Dropout(
            p=dropout
        )

        self.lin = torch.nn.Linear(
            hidden_channels,
            out_channels
        )

    def forward(
        self,
        x_dict,
        edge_index_dict
    ):

        x_dict = self.conv(
            x_dict,
            edge_index_dict
        )

        x_author = x_dict[
            'author'
        ]

        x_author = self.dropout(
            x_author
        )

        out = self.lin(
            x_author
        )

        return out

# Treinamento

## Apoio

In [19]:
def train(model,data,optimizer):
    model.train()
    optimizer.zero_grad()

    out = model(data.x_dict, data.edge_index_dict)

    criterion = torch.nn.CrossEntropyLoss()

    loss = criterion(out[data['author'].train_mask],
                      data['author'].y[data['author'].train_mask])

    loss.backward()
    optimizer.step()

    return loss.item()

In [46]:
@torch.no_grad()
def evaluate(model, data):

    model.eval()

    out = model(
        data.x_dict,
        data.edge_index_dict
    )

    pred = out.argmax(dim=1)

    train_acc = (
        pred[data['author'].train_mask]
        ==
        data['author'].y[data['author'].train_mask]
    ).float().mean()

    val_acc = (
        pred[data['author'].val_mask]
        ==
        data['author'].y[data['author'].val_mask]
    ).float().mean()

    test_acc = (
        pred[data['author'].test_mask]
        ==
        data['author'].y[data['author'].test_mask]
    ).float().mean()

    return (
        train_acc.item(),
        val_acc.item(),
        test_acc.item()
    )

## Baseline

In [45]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data = dblp.to(device)

model = HAN(
    in_channels=-1,  # PyG infere automaticamente
    hidden_channels=64,
    out_channels=4,  # DBLP tem 4 classes de autores
    metadata=metadata,
    heads=8
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=0.001)

In [ ]:
for epoch in range(1, 101):
    loss = train(model,data,optimizer)
    acc = evaluate(model,data)

    print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, Acc: {acc:.4f}")

Epoch 001, Loss: 1.3948, Acc: 0.4105
Epoch 002, Loss: 1.3614, Acc: 0.5570
Epoch 003, Loss: 1.3223, Acc: 0.6033
Epoch 004, Loss: 1.2740, Acc: 0.6257
Epoch 005, Loss: 1.2194, Acc: 0.6371
Epoch 006, Loss: 1.1600, Acc: 0.6472
Epoch 007, Loss: 1.0962, Acc: 0.6641
Epoch 008, Loss: 1.0286, Acc: 0.6779
Epoch 009, Loss: 0.9583, Acc: 0.6905
Epoch 010, Loss: 0.8865, Acc: 0.7006
Epoch 011, Loss: 0.8143, Acc: 0.7080
Epoch 012, Loss: 0.7431, Acc: 0.7169
Epoch 013, Loss: 0.6740, Acc: 0.7277
Epoch 014, Loss: 0.6079, Acc: 0.7353
Epoch 015, Loss: 0.5455, Acc: 0.7452
Epoch 016, Loss: 0.4875, Acc: 0.7538
Epoch 017, Loss: 0.4341, Acc: 0.7651
Epoch 018, Loss: 0.3856, Acc: 0.7753
Epoch 019, Loss: 0.3419, Acc: 0.7826
Epoch 020, Loss: 0.3031, Acc: 0.7885
Epoch 021, Loss: 0.2690, Acc: 0.7967
Epoch 022, Loss: 0.2392, Acc: 0.8017
Epoch 023, Loss: 0.2135, Acc: 0.8038
Epoch 024, Loss: 0.1915, Acc: 0.8081
Epoch 025, Loss: 0.1727, Acc: 0.8084
Epoch 026, Loss: 0.1566, Acc: 0.8075
Epoch 027, Loss: 0.1429, Acc: 0.8066
E

## Variantes

In [25]:
def generate_han_variants():

    variants = []

    for hidden_channels in [32, 64, 128]:

        for heads in [2, 4, 8]:

            for dropout in [0.3, 0.5, 0.7]:

                variants.append({

                    "hidden_channels":
                        hidden_channels,

                    "heads":
                        heads,

                    "dropout":
                        dropout

                })

    return variants

In [48]:
SAVE_DIR = "C:\\faculdade\\Tabalho-Final-XAI\\top_models"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

results = []

dataset_name = "DBLP"

print(f"\n{'='*50}")
print(f"Dataset: {dataset_name}")
print(f"{'='*50}")

data = dblp_new

variants = generate_han_variants()

for variant_id, config in enumerate(variants):

    print(
        f"\nVariant {variant_id}"
    )

    model = HAN(
        in_channels={

            node_type:
            data[node_type].num_features

            for node_type in data.node_types

            if 'x' in data[node_type]

        },

        metadata=data.metadata(),

        hidden_channels=config["hidden_channels"],

        out_channels=4,

        heads=config["heads"],

        dropout=config["dropout"]

    ).to(device)

    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=0.005,

        weight_decay=5e-4

    )

    best_val = 0
    best_test = 0

    for epoch in range(1, 201):

        loss = train(
            model,
            data,
            optimizer
        )

        train_acc, val_acc, test_acc = evaluate(
            model,
            data
        )

        if val_acc > best_val:

            best_val = val_acc
            best_test = test_acc

    result = {

        "dataset": dataset_name,

        "model": "HAN",

        "variant_id": variant_id,

        "hidden_channels":
            config["hidden_channels"],

        "heads":
            config["heads"],

        "dropout":
            config["dropout"],

        "best_val":
            best_val,

        "best_test":
            best_test

    }

    results.append(result)

    print(
        f"Variant {variant_id} | "
        f"Val={best_val:.4f} | "
        f"Test={best_test:.4f}"
    )



Dataset: DBLP

Variant 0
Variant 0 | Val=0.7850 | Test=0.8014

Variant 1
Variant 1 | Val=0.7775 | Test=0.8041

Variant 2
Variant 2 | Val=0.7925 | Test=0.8078

Variant 3
Variant 3 | Val=0.7900 | Test=0.8075

Variant 4
Variant 4 | Val=0.7925 | Test=0.8124

Variant 5
Variant 5 | Val=0.7800 | Test=0.8103

Variant 6
Variant 6 | Val=0.7925 | Test=0.8020

Variant 7
Variant 7 | Val=0.7950 | Test=0.8096

Variant 8
Variant 8 | Val=0.7950 | Test=0.8099

Variant 9
Variant 9 | Val=0.7825 | Test=0.8035

Variant 10
Variant 10 | Val=0.7800 | Test=0.8121

Variant 11
Variant 11 | Val=0.7950 | Test=0.8142

Variant 12
Variant 12 | Val=0.7850 | Test=0.8010

Variant 13
Variant 13 | Val=0.7925 | Test=0.8133

Variant 14
Variant 14 | Val=0.7900 | Test=0.8115

Variant 15
Variant 15 | Val=0.7925 | Test=0.8075

Variant 16
Variant 16 | Val=0.7900 | Test=0.8112

Variant 17
Variant 17 | Val=0.7925 | Test=0.8201

Variant 18
Variant 18 | Val=0.7850 | Test=0.8118

Variant 19
Variant 19 | Val=0.7800 | Test=0.8112

Vari

In [53]:
results_df = pd.DataFrame(results)
results_df

,dataset,model,variant_id,hidden_channels,heads,dropout,best_val,best_test
0,DBLP,HAN,0,32,2,0.3,0.7850,0.801351
1,DBLP,HAN,1,32,2,0.5,0.7775,0.804114
2,DBLP,HAN,2,32,2,0.7,0.7925,0.807799
3,DBLP,HAN,3,32,4,0.3,0.7900,0.807492
4,DBLP,HAN,4,32,4,0.5,0.7925,0.812404
5,DBLP,HAN,5,32,4,0.7,0.7800,0.810255
6,DBLP,HAN,6,32,8,0.3,0.7925,0.801965
7,DBLP,HAN,7,32,8,0.5,0.7950,0.809641
8,DBLP,HAN,8,32,8,0.7,0.7950,0.809948
9,DBLP,HAN,9,64,2,0.3,0.7825,0.803500


In [50]:
top_models = (

    results_df

    .sort_values(

        by="best_test",

        ascending=False

    )

    .head(3)

)

In [52]:
for _, row in top_models.iterrows():

    filename = (

        f"DBLP_HAN"

        f"_h{row['hidden_channels']}"

        f"_heads{row['heads']}"

        f"_d{row['dropout']}"

        ".pth"

    )
    
    model = model = HAN(
        in_channels={

            node_type:
            data[node_type].num_features

            for node_type in data.node_types

            if 'x' in data[node_type]

        },

        metadata=data.metadata(),

        hidden_channels=row["hidden_channels"],

        out_channels=4,

        heads=row["heads"],

        dropout=row["dropout"]

    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.01,
        weight_decay=5e-4
    )

    for epoch in range(200):

        train(
            model,
            data,
            optimizer
        )

    torch.save(

        model.state_dict(),

        os.path.join(
            SAVE_DIR,
            filename
        )

    )

    print(
        f"Modelo salvo: {filename}"
    )

Modelo salvo: DBLP_HAN_h64_heads8_d0.7.pth
Modelo salvo: DBLP_HAN_h128_heads8_d0.7.pth
Modelo salvo: DBLP_HAN_h128_heads2_d0.7.pth


# Conformidade

In [19]:
TOP_DIR = "C:\\faculdade\\Tabalho-Final-XAI\\top_models"

### Dados

In [20]:
dblp = DBdataset[0]


dblp_new = HeteroData()

# nós
dblp_new['author'].x = dblp['author'].x
dblp_new['author'].y = dblp['author'].y
dblp_new['author'].train_mask = dblp['author'].train_mask
dblp_new['author'].val_mask = dblp['author'].val_mask
dblp_new['author'].test_mask = dblp['author'].test_mask

dblp_new['paper'].x = dblp['paper'].x

dblp_new['term'].x = dblp['term'].x

dblp_new['author', 'to', 'paper'].edge_index = \
    dblp['author', 'to', 'paper'].edge_index

dblp_new['paper', 'to', 'author'].edge_index = \
    dblp['paper', 'to', 'author'].edge_index

dblp_new['paper', 'to', 'term'].edge_index = \
    dblp['paper', 'to', 'term'].edge_index

dblp_new['term', 'to', 'paper'].edge_index = \
    dblp['term', 'to', 'paper'].edge_index

### Apoio

In [28]:
from itertools import combinations
import pandas as pd
@torch.no_grad()
def pairwise_agreement(
    models,
    data,
    mask
):

    predictions = {}

    for model_name, model in models.items():

        model.eval()

        out = model(

            data.x_dict,

            data.edge_index_dict

        )

        predictions[
            model_name
        ] = out.argmax(dim=1).cpu()

    results = []

    model_ids = list(
        predictions.keys()
    )

    for model_a, model_b in combinations(
        model_ids,
        2
    ):

        pred_a = predictions[
            model_a
        ][mask.cpu()]

        pred_b = predictions[
            model_b
        ][mask.cpu()]

        agreement = (

            pred_a == pred_b

        ).float().mean().item()

        results.append({

            "model_a":
                model_a,

            "model_b":
                model_b,

            "agreement":
                agreement

        })

    return pd.DataFrame(
        results
    )

In [29]:
from itertools import combinations
import pandas as pd
@torch.no_grad()
def full_agreement_nodes(
    models,
    data,
    mask
):

    predictions = []

    for model in models.values():

        model.eval()

        out = model(

            data.x_dict,

            data.edge_index_dict

        )

        predictions.append(

            out.argmax(dim=1).cpu()

        )

    agreement_mask = torch.ones_like(

        predictions[0],

        dtype=torch.bool

    )

    for pred in predictions[1:]:

        agreement_mask &= (

            pred == predictions[0]

        )

    agreement_mask &= mask.cpu()

    return agreement_mask

In [30]:
def load_han_model(
    hidden_channels,
    heads,
    dropout,
    data,
    device,
    save_dir
):

    model = HAN(

        in_channels={

            node_type:
            data[node_type].num_features

            for node_type in data.node_types

            if 'x' in data[node_type]

        },

        metadata=data.metadata(),

        hidden_channels=hidden_channels,

        out_channels=4,

        heads=heads,

        dropout=dropout

    ).to(device)

    filename = (

        f"DBLP_HAN"

        f"_h{hidden_channels}"

        f"_heads{heads}"

        f"_d{dropout}.pth"

    )

    path = os.path.join(
        save_dir,
        filename
    )

    model.load_state_dict(

        torch.load(
            path,
            map_location=device
        )
    )

    model.eval()

    return model

In [31]:
def load_han_models(
    save_dir,
    data,
    device
):

    models = {}

    for filename in os.listdir(save_dir):

        if not filename.endswith(".pth"):
            continue

        if not filename.startswith("DBLP_HAN"):
            continue

        parts = filename.replace(
            ".pth",
            ""
        ).split("_")

        hidden_channels = int(
            parts[2][1:]
        )

        heads = int(
            parts[3].replace(
                "heads",
                ""
            )
        )

        dropout = float(
            parts[4][1:]
        )

        model = HAN(

            in_channels={

                node_type:
                data[node_type].num_features

                for node_type in data.node_types

                if 'x' in data[node_type]

            },

            metadata=data.metadata(),

            hidden_channels=hidden_channels,

            out_channels=4,

            heads=heads,

            dropout=dropout

        ).to(device)

        model.load_state_dict(

            torch.load(

                os.path.join(
                    save_dir,
                    filename
                ),

                map_location=device

            )

        )

        model.eval()

        models[
            filename
        ] = model

    return models

### DBLP

HAN

In [26]:
han_models = load_han_models(

    TOP_DIR,

    dblp_new,

    device

)

In [27]:
han_models.keys()

dict_keys(['DBLP_HAN_h128_heads2_d0.7.pth', 'DBLP_HAN_h128_heads8_d0.7.pth', 'DBLP_HAN_h64_heads8_d0.7.pth'])

In [32]:
agreement_df = pairwise_agreement(

    han_models,

    dblp_new,

    dblp_new["author"].test_mask

)

agreement_df

,model_a,model_b,agreement
0,DBLP_HAN_h128_heads2_d0.7.pth,DBLP_HAN_h128_heads8_d0.7.pth,0.903285
1,DBLP_HAN_h128_heads2_d0.7.pth,DBLP_HAN_h64_heads8_d0.7.pth,0.898373
2,DBLP_HAN_h128_heads8_d0.7.pth,DBLP_HAN_h64_heads8_d0.7.pth,0.957016


In [35]:
agreement_mask = full_agreement_nodes(

    han_models,

    dblp_new,

    dblp_new["author"].test_mask

)
agreement_rate = (

    agreement_mask.sum().item()

    /

    dblp_new["author"].test_mask.sum().item()

)

print(
    f"Concordância total: {agreement_rate:.2%}"
)

Concordância total: 88.09%
